# データ前処理プログラム - Jupyter Notebook使用例

このノートブックは、JupyterLabやGoogle Colabでデータ前処理プログラムを使用する例です。


## 1. 必要なライブラリのインストール（Google Colabの場合）

ローカルのJupyterLabでは、`pip install -r requirements.txt`を実行してください。


In [ ]:
# Google Colabで使用する場合のみ実行
# !pip install pandas numpy scikit-learn scipy openpyxl


## 2. モジュールのインポート


In [ ]:
from data_preprocessor import DataPreprocessor
import pandas as pd
import numpy as np


## 3. サンプルデータの作成（実際の使用時は、このセルを削除して実際のデータを読み込んでください）


In [ ]:
# サンプルデータを作成（整数型のカテゴリ変数を含む）
np.random.seed(42)
n_samples = 200

data = {
    'age': np.random.randint(20, 80, n_samples),  # 数値変数
    'income': np.random.normal(50000, 15000, n_samples),  # 数値変数
    'city': np.random.choice(['Tokyo', 'Osaka', 'Kyoto'], n_samples),  # 文字列カテゴリ
    'education': np.random.choice(['High School', 'Bachelor', 'Master'], n_samples),  # 文字列カテゴリ
    'gender': np.random.choice([0, 1], n_samples),  # 整数カテゴリ（0=女性, 1=男性）
    'status': np.random.choice([0, 1, 2], n_samples),  # 整数カテゴリ（0=未処理, 1=処理中, 2=完了）
    'experience': np.random.randint(0, 30, n_samples),  # 数値変数
    'target': np.random.choice([0, 1], n_samples)  # ターゲット変数
}

df = pd.DataFrame(data)

# 意図的に欠損値を追加
missing_indices = np.random.choice(df.index, size=int(n_samples * 0.1), replace=False)
df.loc[missing_indices, 'income'] = np.nan

print("元のデータ:")
print(df.head())
print("\nデータ情報:")
print(df.info())
print("\n各列のユニーク値数:")
for col in df.columns:
    print(f"{col}: {df[col].nunique()}個")


## 4. カテゴリ変数の自動識別（改善版 - 整数型のカテゴリ変数も識別）


In [ ]:
preprocessor = DataPreprocessor()

# カテゴリ変数を自動識別（整数型のカテゴリ変数も含む）
numerical_cols, categorical_cols = preprocessor.identify_columns(df, categorical_threshold=10)

print("識別された数値列:")
print(numerical_cols)
print("\n識別されたカテゴリ列:")
print(categorical_cols)
print("\n各カテゴリ列のユニーク値:")
for col in categorical_cols:
    print(f"{col}: {sorted(df[col].unique())}")


## 5. 明示的にカテゴリ変数を指定する場合


In [ ]:
# 特定の列を明示的にカテゴリ変数として指定
preprocessor = DataPreprocessor()
numerical_cols, categorical_cols = preprocessor.identify_columns(
    df, 
    categorical_threshold=10,
    explicit_categorical=['experience']  # experienceをカテゴリ変数として扱う
)

print("数値列:", numerical_cols)
print("カテゴリ列:", categorical_cols)


## 6. 完全な前処理パイプラインの実行


## 7. 日本語ファイルの読み込み（文字化け対策）


In [ ]:
# 日本語を含むデータを作成
df_japanese = df.copy()
df_japanese['都道府県'] = np.random.choice(['東京都', '大阪府', '京都府'], len(df_japanese))
df_japanese['職業'] = np.random.choice(['エンジニア', 'デザイナー', 'マーケター'], len(df_japanese))

# Shift-JISで保存（Windowsでよく使われる形式）
df_japanese.to_csv('japanese_data.csv', index=False, encoding='shift_jis')
print("日本語データを保存しました")

# 自動エンコーディング検出で読み込み
df_loaded = preprocessor.load_data('japanese_data.csv', auto_detect_encoding=True)
print("\n読み込んだデータ:")
print(df_loaded.head())


## 8. 記述統計量の表示


In [ ]:
# 記述統計量を表示
preprocessor.identify_columns(df)
preprocessor.describe_statistics(df, include_all=True)


## 9. データの可視化（日本語フォント自動設定）


In [ ]:
# データの可視化（日本語フォントが自動設定されます）
preprocessor.visualize_data(df)


## 10. 相関行列の可視化


In [ ]:
# 相関行列の可視化（数値変数が2つ以上ある場合）
if len(preprocessor.numerical_columns) >= 2:
    preprocessor.visualize_correlation(df)


## 11. 欠損値の可視化


In [ ]:
# 欠損値の可視化
if df.isnull().sum().sum() > 0:
    preprocessor.visualize_missing_values(df)
else:
    print("欠損値はありません。")


## 12. 包括的な分析（統計量 + 可視化）


## 13. Isolation Forestによる外れ値検出


In [ ]:
# Isolation Forestによる外れ値検出
preprocessor.identify_columns(df)

print("元のデータ形状:", df.shape)

# IQR法
df_iqr = preprocessor.remove_outliers(df, method='iqr')
print(f"IQR法後のデータ形状: {df_iqr.shape}")

# Z-score法
df_zscore = preprocessor.remove_outliers(df, method='zscore')
print(f"Z-score法後のデータ形状: {df_zscore.shape}")

# Isolation Forest
df_iso = preprocessor.remove_outliers(df, method='isolation_forest', contamination=0.1)
print(f"Isolation Forest後のデータ形状: {df_iso.shape}")


## 14. 統計検定


In [ ]:
# 正規性検定のみ
normality_results = preprocessor.test_normality(df, alpha=0.05)

# 包括的な統計検定（正規性、等分散性、独立性など）
test_results = preprocessor.statistical_tests(df, alpha=0.05)


In [ ]:
# 記述統計量と可視化を一度に実行
preprocessor.analyze_data(df, visualize=True, save_plots=False)


In [ ]:
# データをCSVに保存（実際の使用時は、この行を削除して実際のファイルパスを指定）
df.to_csv('sample_data.csv', index=False)

# 前処理パイプラインを実行
processed_df = preprocessor.preprocess_pipeline(
    file_path='sample_data.csv',
    target_col='target',
    missing_strategy='auto',
    encoding_method='auto',
    scaling_method='standard',
    remove_outliers_flag=True,
    feature_selection=True,
    k_features=5,
    categorical_threshold=10,  # ユニーク値10以下の整数列をカテゴリ変数として扱う
    explicit_categorical=None  # 必要に応じて明示的に指定
)

print("\n前処理後のデータ:")
print(processed_df.head())
print(f"\nデータ形状: {processed_df.shape}")
